# 1. Research Question

Is greater trading frequency associated with higher profitability? Does profitability exhibit diminishing returns as frequency increases?

The unit of analysis is a trader with at least 100 transactions. The main outcome is **profit per unit of volume (`trader_ppv`)**; total profit/loss (`trader_pnl`) is a secondary outcome. Each trader receives equal weight. This is an observational analysis: associations do not establish causation.

# 2. Data Loading

Run this notebook from the directory containing `data_no_imputation.parquet`. It initializes all its own variables and never writes to the source dataset. Dependencies: pandas, NumPy, Matplotlib, SciPy, and a pandas-compatible Parquet engine (such as PyArrow).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (11, 6), "font.size": 11})
pd.set_option("display.max_columns", 20)

df = pd.read_parquet("data_no_imputation.parquet")
print(f"Dataset shape: {df.shape[0]:,} rows ? {df.shape[1]:,} columns")

numeric_columns = [
    "transactions_per_day", "transaction_count", "trader_ppv",
    "trader_pnl", "trader_volume",
]
required_columns = ["trader", *numeric_columns]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")
print("All required columns exist.")
display(df[required_columns].dtypes.rename("dtype").to_frame())

# 3. Data Preparation

Verify trader identifiers before treating rows as independent trader observations. Stop if identifiers are missing, blank, or repeated; do not silently deduplicate traders. Keep `transaction_count >= 100`, then require finite values in all five numerical columns, including volume. No imputation, trimming, or winsorization is applied. Negative trading frequency is treated as a data-quality error; zero frequency is retained.

In [ ]:
trader_ids = df["trader"].astype("string").str.strip()
missing_ids = trader_ids.isna() | trader_ids.eq("").fillna(False)
duplicate_ids = trader_ids.loc[~missing_ids].duplicated(keep=False)
print(f"Missing/blank trader IDs: {missing_ids.sum():,}")
print(f"Rows with repeated trader IDs: {duplicate_ids.sum():,}")
if missing_ids.any() or duplicate_ids.any():
    raise ValueError("Rows are not verified as unique traders; resolve trader identifiers before analysis.")
print("Verified: every row represents a distinct trader identifier.")

analysis_df = df[required_columns].copy()
for column in numeric_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="raise")

analysis_df = analysis_df.loc[analysis_df["transaction_count"] >= 100].copy()
threshold_count = len(analysis_df)
finite_mask = np.isfinite(
    analysis_df[numeric_columns].to_numpy(dtype=float, na_value=np.nan)
).all(axis=1)
analysis_df = analysis_df.loc[finite_mask].copy()
print(f"Rows meeting transaction-count threshold: {threshold_count:,}")
print(f"Removed for missing/infinite relevant values: {threshold_count - len(analysis_df):,}")
print(f"Traders remaining: {len(analysis_df):,}")
if len(analysis_df) < 3:
    raise ValueError("At least three valid traders are needed for this analysis.")
if (analysis_df["transactions_per_day"] < 0).any():
    raise ValueError("Negative transactions_per_day found; inspect the source data.")
if analysis_df["transactions_per_day"].nunique() < 2:
    raise ValueError("Trading frequency has no variation.")
display(analysis_df[numeric_columns].describe(percentiles=[0.01, 0.5, 0.9, 0.99]))

In [ ]:
# Quantile boundaries preserve equal frequency values in the same group.
# Tied boundaries may yield fewer than 10 groups; never split ties arbitrarily.
frequency_groups, frequency_edges = pd.qcut(
    analysis_df["transactions_per_day"], q=10, retbins=True, duplicates="drop"
)
analysis_df["frequency_group"] = frequency_groups
summary_table = (
    analysis_df.groupby("frequency_group", observed=True)
    .agg(
        n_traders=("trader", "size"),
        median_transactions_per_day=("transactions_per_day", "median"),
        min_transactions_per_day=("transactions_per_day", "min"),
        max_transactions_per_day=("transactions_per_day", "max"),
        mean_trader_ppv=("trader_ppv", "mean"),
        median_trader_ppv=("trader_ppv", "median"),
        mean_trader_pnl=("trader_pnl", "mean"),
        median_trader_pnl=("trader_pnl", "median"),
    )
    .reset_index()
)
summary_table.insert(0, "group", [f"Q{i + 1}" for i in range(len(summary_table))])
assert summary_table["n_traders"].sum() == len(analysis_df)
print(f"Requested 10 quantile groups; obtained {len(summary_table)} nonempty groups.")
if len(summary_table) != 10:
    print("Repeated quantile boundaries or empty intervals reduced the group count.")
print("Group sizes can differ because traders with tied frequencies stay together.")
display(summary_table.round(6))

# 4. Visualizations

Four separate figures use the same filtered sample. The X-axis uses log scaling when the positive frequency range spans at least two orders of magnitude. If zeros are present, a symmetric-log axis preserves them with a linear region near zero. Otherwise a linear axis is used. Axis scaling changes presentation only; it does not remove observations. Profit axes remain linear, including losses and finite extremes.

The grouped charts plot equally weighted trader means/medians against each group's median frequency. Their labels show group identifiers and sample sizes; the summary table gives the frequency ranges.

In [ ]:
frequency = analysis_df["transactions_per_day"].to_numpy(dtype=float)
positive_frequency = frequency[frequency > 0]
positive_span = positive_frequency.max() / positive_frequency.min() if positive_frequency.size else 1.0
zero_count = int((frequency == 0).sum())
frequency_scale = "linear"
if positive_span >= 100:
    frequency_scale = "symlog" if zero_count else "log"
print(f"Frequency range: {frequency.min():.6g} to {frequency.max():.6g} transactions/day")
print(f"Positive max/min ratio: {positive_span:,.1f}; zero-frequency traders: {zero_count:,}")
print(f"Selected X-axis scale: {frequency_scale}")

def format_frequency_axis(ax):
    if frequency_scale == "symlog":
        ax.set_xscale("symlog", linthresh=float(positive_frequency.min()))
    else:
        ax.set_xscale(frequency_scale)
    ax.set_xlabel(f"Average transactions per day ({frequency_scale} scale)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.grid(alpha=0.2)

fig, ax = plt.subplots()
ax.scatter(frequency, analysis_df["trader_ppv"], alpha=0.15, s=12, edgecolors="none")
ax.set_title("Trading frequency vs. profit per volume | traders with 100+ transactions")
ax.set_ylabel("Profit per unit of trading volume (trader_ppv)")
format_frequency_axis(ax)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(summary_table["median_transactions_per_day"], summary_table["mean_trader_ppv"], marker="o")
for i, row in summary_table.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_transactions_per_day"], row["mean_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Mean profit per volume by trading-frequency quantile group")
ax.set_ylabel("Mean trader_ppv (equal trader weights)")
format_frequency_axis(ax)
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(summary_table["median_transactions_per_day"], summary_table["median_trader_ppv"], marker="o")
for i, row in summary_table.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_transactions_per_day"], row["median_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Median profit per volume by trading-frequency quantile group")
ax.set_ylabel("Median trader_ppv")
format_frequency_axis(ax)
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(frequency, analysis_df["trader_pnl"], alpha=0.15, s=12, edgecolors="none")
ax.set_title("Trading frequency vs. total profit/loss | traders with 100+ transactions")
ax.set_ylabel("Total profit/loss (trader_pnl; dataset units)")
format_frequency_axis(ax)
fig.tight_layout()
plt.show()

# 5. Statistical Analysis

Pearson measures linear association; Spearman measures rank association. Both use individual traders and untransformed variables. Report both PPV and total P&L. P-values are nominal, assume independent observations, and are not adjusted for multiple comparisons. Distinct wallet identifiers do not guarantee independent owners. Large samples can produce small p-values for weak associations.

For exploratory nonlinearity, compare an ordinary linear model in raw frequency with linear and quadratic models in `log1p(frequency)`. All fits retain extreme observations and use equal trader weights. In-sample R? is descriptive, not a validation score; a quadratic model can improve R? simply by adding flexibility. Group-to-group slopes provide a second descriptive check on the original frequency scale. No extra figures are created.

In [ ]:
correlation_rows = []
for outcome in ["trader_ppv", "trader_pnl"]:
    values = analysis_df[outcome].to_numpy(dtype=float)
    for method, function in [("Pearson", stats.pearsonr), ("Spearman", stats.spearmanr)]:
        if np.unique(values).size < 2:
            coefficient, p_value = np.nan, np.nan
        else:
            coefficient, p_value = function(frequency, values)
        correlation_rows.append({
            "outcome": outcome, "method": method,
            "correlation": coefficient, "p_value": p_value,
            "n_traders": len(values),
        })
correlation_table = pd.DataFrame(correlation_rows)
display(correlation_table.style.format({"correlation": "{:.6f}", "p_value": "{:.6g}", "n_traders": "{:,.0f}"}))

In [ ]:
ppv = analysis_df["trader_ppv"].to_numpy(dtype=float)
raw_z = (frequency - frequency.mean()) / frequency.std()
log_frequency = np.log1p(frequency)
log_z = (log_frequency - log_frequency.mean()) / log_frequency.std()
model_designs = {
    "Linear in frequency": np.column_stack([np.ones(len(frequency)), raw_z]),
    "Linear in log1p(frequency)": np.column_stack([np.ones(len(frequency)), log_z]),
    "Quadratic in log1p(frequency)": np.column_stack([np.ones(len(frequency)), log_z, log_z**2]),
}
model_rows = []
model_coefficients = {}
total_ss = np.sum((ppv - ppv.mean())**2)
for name, design in model_designs.items():
    coefficients, _, rank, _ = np.linalg.lstsq(design, ppv, rcond=None)
    model_coefficients[name] = coefficients
    residual_ss = np.sum((ppv - design @ coefficients)**2)
    model_rows.append({
        "model": name, "n_traders": len(ppv), "rank": rank,
        "n_parameters": design.shape[1],
        "in_sample_r_squared": 1 - residual_ss / total_ss if total_ss > 0 else np.nan,
        "quadratic_coefficient": coefficients[2] if len(coefficients) == 3 and rank == 3 else np.nan,
    })
model_table = pd.DataFrame(model_rows)
display(model_table.round(8))

# Positive slopes that decrease at higher frequencies would be consistent
# with diminishing gains. Flat, negative, or erratic slopes need other interpretations.
slope_table = summary_table[["group", "median_transactions_per_day", "mean_trader_ppv", "median_trader_ppv"]].copy()
frequency_steps = slope_table["median_transactions_per_day"].diff().replace(0, np.nan)
slope_table["mean_ppv_change_per_extra_transaction_per_day"] = slope_table["mean_trader_ppv"].diff() / frequency_steps
slope_table["median_ppv_change_per_extra_transaction_per_day"] = slope_table["median_trader_ppv"].diff() / frequency_steps
display(slope_table.round(8))
print("Slopes connect adjacent group medians; they are descriptive, not individual-trader marginal effects.")
print("Quadratic curvature is on the log1p frequency scale, not the raw-frequency scale.")

# 6. Key Findings

The following findings are calculated from this run. Assess effect sizes, the mean-versus-median difference, the grouped slopes, and the scatter plots together. Total P&L also reflects trading volume and exposure; this analysis does not control for volume, tenure, strategy, or other confounders.

In [ ]:
ppv_correlations = correlation_table.loc[correlation_table["outcome"] == "trader_ppv"].set_index("method")
first_group, last_group = summary_table.iloc[0], summary_table.iloc[-1]
r_squared = model_table.set_index("model")["in_sample_r_squared"]
curvature = model_table.loc[model_table["model"] == "Quadratic in log1p(frequency)", "quadratic_coefficient"].iloc[0]
mean_slopes = slope_table["mean_ppv_change_per_extra_transaction_per_day"].dropna().to_numpy()
median_slopes = slope_table["median_ppv_change_per_extra_transaction_per_day"].dropna().to_numpy()

def diminishing_pattern(slopes):
    return len(slopes) >= 2 and np.all(slopes > 0) and np.all(np.diff(slopes) <= 0)

pattern = diminishing_pattern(mean_slopes) and diminishing_pattern(median_slopes)
pattern_text = (
    "Both grouped curves have positive, decreasing adjacent slopes, a descriptive pattern consistent with diminishing gains."
    if pattern else
    "The mean and median grouped curves do not both show consistently positive, decreasing adjacent slopes; they do not provide a clean diminishing-returns pattern."
)
display(Markdown(
    f"- **Sample:** {len(analysis_df):,} unique traders with at least 100 transactions; {len(summary_table)} frequency groups.\n"
    f"- **Frequency vs. PPV:** Pearson r = {ppv_correlations.loc['Pearson', 'correlation']:.4f} "
    f"(p = {ppv_correlations.loc['Pearson', 'p_value']:.3g}); "
    f"Spearman rho = {ppv_correlations.loc['Spearman', 'correlation']:.4f} "
    f"(p = {ppv_correlations.loc['Spearman', 'p_value']:.3g}).\n"
    f"- **Lowest to highest frequency group:** mean PPV {first_group['mean_trader_ppv']:.6f} ? {last_group['mean_trader_ppv']:.6f}; "
    f"median PPV {first_group['median_trader_ppv']:.6f} ? {last_group['median_trader_ppv']:.6f}.\n"
    f"- **Nonlinearity:** in-sample R? is {r_squared['Linear in frequency']:.5f} for raw linear, "
    f"{r_squared['Linear in log1p(frequency)']:.5f} for log-linear, and "
    f"{r_squared['Quadratic in log1p(frequency)']:.5f} for log-quadratic. "
    f"The standardized log-frequency quadratic coefficient is {curvature:.6g}. "
    "These descriptive fits alone do not establish a nonlinear population relationship.\n"
    f"- **Diminishing returns:** {pattern_text} Grouping, finite extremes, and sampling variability affect these comparisons.\n"
    "- **Interpretation:** association is not causation. Greater frequency cannot be recommended as a way to improve profit based on these results."
))